In [ ]:
from pyspark.sql.functions import current_timestamp, lit

In [ ]:
storage_key = dbutils.secrets.get(
    scope="your-secret-scope",
    key="azure-storage-key"
)

spark.conf.set(
    "fs.azure.account.key.stgrocerywaste.dfs.core.windows.net",
    storage_key
)


In [ ]:
landing_full_path = "abfss://landing@stgrocerywaste.dfs.core.windows.net/grocery_expiration_waste_1M.csv"

landing_inc_path = "abfss://landing@stgrocerywaste.dfs.core.windows.net/gro_incremental.csv"

bronze_path = "abfss://bronze@stgrocerywaste.dfs.core.windows.net/inventory"

In [ ]:
df_full = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(landing_full_path)

In [ ]:
df_inc = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(landing_inc_path)

In [ ]:
df_raw = df_full.unionByName(df_inc)

In [ ]:
df_bronze = df_raw \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("source_file", lit("landing_data"))

In [ ]:
df_bronze.write \
    .mode("append") \
    .format("delta") \
    .save(bronze_path)